# Parte 3: Implementando un Cifrado de Flujo con ChaCha20

- Implementar ChaCha20 para cifrar y descifrar mensajes.
- Comparar su rendimiento con AES en tiempos y consumo de memoria.

In [4]:
import timeit
from Cryptodome.Cipher import ChaCha20
from AES import *
import memory_profiler

In [5]:
# Encriptado y Decriptado con ChaCha20
def cipherChaCha20(data, key=None, nonce=None):
    if key is None:
        key = getRandomKey(32)
    if nonce is None:
        nonce = getRandomKey(12)

    cipher = ChaCha20.new(key=key, nonce=nonce)
    cipher_data = cipher.encrypt(data)

    return cipher_data, key, nonce

def decipherChaCha20(cipher_data, key, nonce):
    cipher = ChaCha20.new(key=key, nonce=nonce)
    decrypted_data = cipher.decrypt(cipher_data)

    return decrypted_data

In [6]:
# Leer archivo de prueba
message = ""

with open("tests/test_file.txt", "r", encoding="utf-8") as file:
    message = file.read().encode("utf-8")

In [7]:
# Parámetros iniciales
key_chacha = getRandomKey(32)
nonce_chacha = getRandomKey(8)
key_aes = getRandomKey(16)
iv_aes = getRandomKey(AES.block_size)

# Textos encriptados
cipherChaCha, _, _ = cipherChaCha20(message, key_chacha, nonce_chacha)
cipherAES, _, _ = cipherAES_CBC(message, key_aes, iv_aes)

In [8]:
# Comparación de tiempos
time_chacha_encrypt = timeit.timeit(lambda: cipherChaCha20(message, key_chacha, nonce_chacha), number=100)
time_aes_encrypt = timeit.timeit(lambda: cipherAES_CBC(message, key_aes, iv_aes), number=100)

time_chacha_decrypt = timeit.timeit(lambda: decipherChaCha20(cipherChaCha, key_chacha, nonce_chacha), number=100)
time_aes_decrypt = timeit.timeit(lambda: decipherAES_CBC(cipherAES, key_aes, iv_aes), number=100)

print("=========== Comparación de Tiempos ===========")
print(f"Tiempo de Cifrado con ChaCha20: {time_chacha_encrypt:.6f} s")
print(f"Tiempo de Cifrado con AES: {time_aes_encrypt:.6f} s\n")

print(f"Tiempo de Descifrado con ChaCha20: {time_chacha_decrypt:.6f} s")
print(f"Tiempo de Descifrado con  AES: {time_aes_decrypt:.6f} s")

=========== Comparación de Tiempos ===========
Tiempo de Cifrado con ChaCha20: 476.116012 s
Tiempo de Cifrado con AES: 254.551335 s

Tiempo de Descifrado con ChaCha20: 443.715740 s
Tiempo de Descifrado con  AES: 225.042695 s


In [9]:
# Comparación de uso de Memoria
memory_chacha_encrypt = memory_profiler.memory_usage((cipherChaCha20, (message, key_chacha, nonce_chacha)))
memory_aes_encrypt = memory_profiler.memory_usage((cipherAES_CBC, (message, key_aes, iv_aes)))

memory_chacha_decrypt = memory_profiler.memory_usage((decipherChaCha20, (cipherChaCha, key_chacha, nonce_chacha)))
memory_aes_decrypt = memory_profiler.memory_usage((decipherAES_CBC, (cipherAES, key_aes, iv_aes)))

print("=========== Comparación de uso de memoria ===========\n")
print(f"Memoria utilizada para Cifrar con ChaCha20: {max(memory_chacha_encrypt) - min(memory_chacha_encrypt):.6f} MiB")
print(f"Memoria utilizada para Cifrar con AES: {max(memory_aes_encrypt) - min(memory_aes_encrypt):.6f} MiB\n")

print(f"Memoria utilizada para Descifrar con ChaCha20: {max(memory_chacha_decrypt) - min(memory_chacha_decrypt):.6f} MiB")
print(f"Memoria utilizada para Descifrar con AES: {max(memory_aes_decrypt) - min(memory_aes_decrypt):.6f} MiB")

=========== Comparación de uso de memoria ===========

Memoria utilizada para Cifrar con ChaCha20: 1767.851562 MiB
Memoria utilizada para Cifrar con AES: 3012.527344 MiB

Memoria utilizada para Descifrar con ChaCha20: 1719.828125 MiB
Memoria utilizada para Descifrar con AES: 1908.304688 MiB


## Preguntas para reflexión:

### ¿Analizar que cifrado es mas rápido ChaCha20 o AES?

R: En este ejercicio, el algoritmo AES con el modo de operación CBC resultó más rápido que ChaCha20 tanto en cifrado como en descifrado. Para un archivo de 1 GB, repetido una cantidad de 100 veces, AES-CBC tomó 254.55 segundos para cifrar y 225.04 segundos para descifrar, mientras que ChaCha20 tomó 476.12 segundos para cifrar y 443.72 segundos para descifrar. Esto se debe en parte a que AES está optimizado en hardware en muchas CPUs modernas, lo que acelera su ejecución. ChaCha20, al ser un cifrado basado en software, no tiene la misma aceleración hardware. Sin embargo, ChaCha20 demostró un uso de memoria más eficiente, utilizando 1767.85 MiB para cifrar y 1719.83 MiB para descifrar, en comparación con los 3012.53 MiB y 1908.30 MiB utilizados por AES-CBC.

### ¿En qué casos debería usarse en vez de AES?

R: ChaCha20 es una opción favorable en entornos donde los recursos de memoria son limitados. Aunque es más lento que AES-CBC en este ejercicio, su eficiencia en el uso de memoria lo hace adecuado para sistemas con restricciones de hardware. Además, ChaCha20 es resistente a ciertos tipos de ataques, como los ataques de canal lateral basados en tiempo, lo que lo convierte en una opción segura en entornos donde la seguridad es crítica. También es más fácil de implementar correctamente en software, ya que no requiere manejo de modos de operación complejos como en AES. Por lo tanto, ChaCha20 es una excelente alternativa en situaciones donde la eficiencia de memoria y la facilidad de implementación son prioritarias.